# 03 — Grad-CAM et weak localization : validation explicabilite avant integration

**Auteur :** smatt  
**Statut :** notebook de verification qualitative avant exposition applicative

Ce notebook me sert a controler un point simple mais critique : quand le modele produit une prediction forte, regarde-t-il une zone anatomiquement plausible ou exploite-t-il des artefacts du dataset ?

## Usage attendu

- charger un modele exporte ;
- tester plusieurs images representatives et non une seule image flatteuse ;
- verifier la coherence visuelle des heatmaps ;
- documenter les cas ou l'attention semble se fixer sur le texte, les bords, les marqueurs ou le fond.

## Limite

Grad-CAM ne prouve pas une causalite clinique. C'est un outil de revue qualitative pour eliminer les modeles trompeurs avant toute discussion de pre-production.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

from src.preprocessing.image_loader import load_and_preprocess_image


In [ ]:
MODEL_PATH = Path('artifacts/models/optimized_model.keras')
IMAGE_PATH = None

candidates = list(Path('data').rglob('*.png')) + list(Path('data').rglob('*.jpg')) + list(Path('data').rglob('*.jpeg'))
if candidates:
    IMAGE_PATH = candidates[0]

print('Model exists:', MODEL_PATH.exists(), MODEL_PATH)
print('Sample image:', IMAGE_PATH)


In [ ]:
def make_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


In [ ]:
if MODEL_PATH.exists() and IMAGE_PATH is not None:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    x = load_and_preprocess_image(IMAGE_PATH, image_size=224)
    batch = np.expand_dims(x, 0)

    conv_layers = [layer.name for layer in model.layers if isinstance(layer, tf.keras.layers.Conv2D)]
    print('Dernières couches conv candidates:', conv_layers[-5:])
    last_conv = conv_layers[-1]
    heatmap = make_gradcam_heatmap(model, batch, last_conv)

    original = np.asarray(Image.open(IMAGE_PATH).convert('RGB').resize((224, 224)))
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.axis('off')
    plt.title('Image')

    plt.subplot(1, 2, 2)
    plt.imshow(original)
    plt.imshow(heatmap, cmap='jet', alpha=0.35)
    plt.axis('off')
    plt.title('Grad-CAM overlay')
    plt.tight_layout()
else:
    print('Ajoute un modèle Keras exporté et une image pour exécuter ce notebook de bout en bout.')


## Interpretation et criteres de validation

Ce notebook devient utile seulement si la revue est systematique. En pratique, il faut verifier un petit lot de cas bien classes, mal classes et limites, puis noter si la carte d'activation cible la zone pertinente.

### Signaux d'alerte

- activation concentree sur les coins, le texte ou les bordures ;
- heatmaps tres diffuses, sans region saillante interpretable ;
- dependance forte a une seule couche convolutionnelle choisie par commodite ;
- incoherence manifeste entre prediction forte et region active.

### Decision

Si ces alertes apparaissent frequemment, le modele ne doit pas etre promu vers l'API ou la demonstration utilisateur tant qu'une revue d'erreurs et un nouveau cycle d'entrainement n'ont pas ete menes.